# Track B: NYC TLC Operational Dependability Pipeline

This notebook narrates the batch pipeline implemented in `run_pipeline.py`. It follows the path from two raw TLC files to a monthly, auditable metric row.

## Sources and Retrieval

- Trip events: one completed yellow-taxi trip per Parquet row, downloaded for a requested month from the NYC TLC CloudFront endpoint.
- Zone reference: one TLC location per CSV row; joins trip pickup/dropoff IDs to borough names.

Downloads are streamed to temporary `.part` files and promoted after HTTP success and byte-count checks where `Content-Length` is available. Existing non-empty raw files are reused.

## Validation and Workflow Model

The pipeline derives trip duration and average speed, then creates `valid_trips` and `quarantined_trips`. Rules require duration 1-360 minutes, distance greater than 0 and less than 200 miles, nonnegative fare and total, known location IDs, and speed no greater than 80 mph. Quarantine records include a reason and are exported by reporting month.

The fact grain is a completed trip. Zone lookup rows enrich pickup/dropoff boroughs for metric aggregation. The source map documents the workflow states and relational model.

## Metrics and Dependability

The monthly output includes validity rate, Manhattan-to-Queens P95 duration, fare per active minute, and peak/off-peak duration ratio. Re-running a month replaces that period's metric row. The generated CSV is under `data/processed/`; quarantine output and execution logs are retained separately.

Thresholds are explicit assumptions. In particular, Manhattan-to-Queens is a broad corridor proxy, not proof that a trip served an airport. See `evidence/metrics-evidence.md` for the Known / Unknown / Assumption / Limitation table and actual-value evidence instructions.

For the complete source ownership map, workflow diagram, and data model, see `nyc-tlc-pipeline-source-mapping.md`. For the 3-5 minute presentation, see `docs/demo-script.md`.

In [ ]:
from pathlib import Path

from run_pipeline import run

# The CLI module is the single executable source of truth for the pipeline.
metrics = run("2024-01", Path("."))
display(metrics)